# P2P / DyNECT — grouped clearing, stronger curvature (`1e-3`, `1e-2`)

**Standalone experiment, 2016-01-07 · four prosumers · six 4-hour time steps.** Run cells in order from the `p2p_dynect` folder or the original P2P repo.

This notebook preserves the device model and the previously screened feeder/local constraints. It varies **only**:

1. The existing `input_regularization` coefficient in the LQ objective: baseline `1e-9`, comparison `1e-4`, and tests `1e-3` and `1e-2`.
2. The shared P2P clearing representation: exact four-edge bilateral clearing versus the **one-community, ±0.01 kW grouped clearing** with bilateral mismatch penalty `ρ = 100`.

It prints both **conditioning** (`min eig(H+Hᵀ)`, Cholesky, condition estimate) and **economic fidelity** (individual day-1 costs, dispatch and bilateral mismatch). The last cell invokes unmodified `GNESetSolver` in a *separate process with a time limit only*. It never patches Julia and never uses a RAM watchdog.

**Interpretation:** increasing the ridge alters game objectives, and grouping trade clearing alters the admissible game. Matching a vGNE does not imply matching the entire GNE set or its fairest member.

## 1 · Load source scenario and model helpers (no optimization yet)

In [1]:
import sys, copy, math, time, itertools
from pathlib import Path
from dataclasses import replace

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from IPython.display import display

DATE = "2016-01-07"
SEED = 42
SOURCE_DT = 0.5
TARGET_DT = 4.0

# Look upward for the original project, even if notebook lives in p2p_dynect/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "scenarios.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot find scenarios.py. Open notebook in your P2P repo or its p2p_dynect subfolder.")

# Favor an existing p2p package in the notebook's directory, then original imports.
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))
DYNECT_REPO = Path.home() / "Documents/Github/dynect_py_p2p"
if DYNECT_REPO.is_dir():
    sys.path.insert(0, str(DYNECT_REPO))

import config
from scenarios import make_scenario_horizon
from dynect import DynLQGameTV
from p2p.setup import (
    build_game, load_ieee33, _feeder_maps, unpack_controls,
    evaluated_costs, feeder_results, rollout_states,
)
from p2p.vgne import solve_vgne

print("Original P2P repo:", ROOT)
print("DyNECT source:", __import__("dynect").__file__)
print("P2P adapter source:", __import__("p2p.setup", fromlist=["x"]).__file__)
print("Experiment:", DATE, "population:", config.PROSUMER_SCENARIO)

# Build the source day once, without the 48-hour lookahead.
# (This does not run any optimization.)
short_data = make_scenario_horizon(
    DATE, seed=SEED, resolution_h=SOURCE_DT, lookahead_h=0
)
grid = load_ieee33()
assert len(short_data["buy_price"]) == 48, "Expected 24h of half-hour source data"
print("Source:", len(short_data["buy_price"]), "steps ×", short_data["dt"], "h")


Original P2P repo: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative
DyNECT source: /Users/catharinahente/Documents/Github/dynect_py_p2p/dynect/__init__.py
P2P adapter source: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/p2p/setup.py
Experiment: 2016-01-07 population: 4
Building winter scenario (T=48, dt=0.5h, resolution=30min)...
  Prices 2016: loaded from cache
  P1: Residential ++ (bus 5)
    HP: Air_Alternative_1 (Air alternative, Hanover)
  P2: Office building (bus 11)
    ⚠ No EV activity detected — using fallback schedule
  P3: Residential HP only (bus 17)
    HP: Air_Semi-Parallel_1 (Air semi-parallel, Hanover)
  P4: Residential PV only (bus 32)
  Done. N=4, T=48, dt=0.5h, max_demand=16.3 kW
Source: 48 steps × 0.5 h


In [6]:
def clean_local(A, b):
    A = np.asarray(A, float)
    b = np.asarray(b, float)
    best = {}
    for row, rhs in zip(A, b):
        row = row.copy()
        row[row == 0.] = 0.  # normalize -0.0, no numerical rounding
        if not np.any(row):
            if rhs < 0:
                raise ValueError('Infeasible identically-zero local row')
            continue
        key = row.tobytes()   # only EXACT identical normals
        if key not in best or rhs < best[key][1]:
            best[key] = (row, rhs)
    if not best:
        return np.zeros((0, A.shape[1])), np.zeros(0)
    return np.vstack([v[0] for v in best.values()]), np.array([v[1] for v in best.values()])

GAME_FIELDS = ['A','B','Q','R','P','c','q','r','p',
               'C_x','b_x','C_loc','b_loc','C_u','b_u']

def rebuild_constraints(base, *, keep_labels=None, clean_locals=False,
                        localize_single_agent=False):
    """Create a new DynLQGameTV, preserving dynamics/costs and control ordering.

    Shared rows can be removed by their labels (after an independent proof of
    redundancy), or moved to local constraints when they involve one agent.
    """
    old = base.game
    kw = {k: getattr(old,k) for k in GAME_FIELDS}
    labels = list(base.shared_labels)
    if keep_labels is None:
        keep_labels = labels
    if len(set(keep_labels)) != len(keep_labels):
        raise ValueError('Shared label names must be unique')
    indices = [j for j,label in enumerate(labels) if label in set(keep_labels)]
    if len(indices) != len(keep_labels):
        raise ValueError('Unknown requested shared label')
    c_loc, b_loc, c_u, b_u = [], [], [], []
    kept_labels = [labels[j] for j in indices]
    moved = []
    ownership = {}
    if localize_single_agent:
        for j in indices:
            owners = [i for i in range(base.N) if any(
                np.any(old.C_u[t][i][j] != 0) for t in range(base.T))]
            ownership[j] = owners
            if len(owners) == 1:
                moved.append(j)
            elif len(owners) == 0:
                if any(old.b_u[t][j] < 0 for t in range(base.T)):
                    raise ValueError(f'Constant infeasible shared row: {labels[j]}')
                moved.append(j)
        indices = [j for j in indices if j not in moved]
        kept_labels = [labels[j] for j in indices]
    for t in range(base.T):
        local_a = [np.asarray(old.C_loc[t][i]).copy() for i in range(base.N)]
        local_b = [np.asarray(old.b_loc[t][i]).copy() for i in range(base.N)]
        if localize_single_agent:
            for j in moved:
                owners = ownership[j]
                if len(owners) == 1:
                    i = owners[0]
                    local_a[i] = np.vstack((local_a[i], old.C_u[t][i][j:j+1]))
                    local_b[i] = np.append(local_b[i], old.b_u[t][j])
        if clean_locals:
            cleaned = [clean_local(a,b) for a,b in zip(local_a,local_b)]
            local_a = [x[0] for x in cleaned]
            local_b = [x[1] for x in cleaned]
        c_loc.append(local_a); b_loc.append(local_b)
        c_u.append([old.C_u[t][i][indices,:].copy() for i in range(base.N)])
        b_u.append(old.b_u[t][indices].copy())
    kw.update(C_loc=c_loc, b_loc=b_loc, C_u=c_u, b_u=b_u)
    new = replace(base, game=DynLQGameTV(**kw), shared_labels=kept_labels)
    return new, [labels[j] for j in moved]

def model_max_violation(case, u):
    """Check a candidate against the given (possibly FULL) dynamic game's rows,
    without constructing an additional enormous condensed AVI.
    """
    controls = unpack_controls(case, u)
    states = rollout_states(case, u)
    worst = -np.inf
    for t in range(case.T):
        shared = np.zeros(len(case.game.b_u[t]))
        for i in range(case.N):
            ui = np.array([controls[i][name][t] for name in case.controls[i]])
            shared += case.game.C_u[t][i] @ ui
            A = case.game.C_loc[t][i]
            if A.shape[0]:
                worst = max(worst,float(np.max(A @ ui - case.game.b_loc[t][i])))
        if shared.size:
            worst = max(worst,float(np.max(shared-case.game.b_u[t])))
        if case.game.C_x[t].shape[0]:
            worst = max(worst,float(np.max(case.game.C_x[t]@states[t+1]-case.game.b_x[t])))
    return max(0.,float(worst))

In [7]:
def injection_intervals(data):
    N, T = len(data['demand']), len(data['buy_price'])
    lo, hi = np.zeros((N,T)), np.zeros((N,T))
    base = np.asarray(data['generation'],float) - np.asarray(data['demand'],float)
    available = np.asarray(data['ev_availability'],float)
    for i in range(N):
        batt_limit = min(float(data['battery_power'][i]),
                         float(data['battery_smax'][i]) *
                         math.cos(math.pi/float(data.get('battery_polygon_sides',8))))
        ev_limit = min(float(data['ev_power'][i]),
                       float(data['ev_smax'][i]) *
                       math.cos(math.pi/float(data.get('ev_polygon_sides',8))))
        ev_ch = ev_limit*available[i]
        ev_dis = ev_ch * float(bool(data.get('ev_v2g',True)))
        lo[i] = base[i] - batt_limit - ev_ch - float(data['hp_max_power'][i])
        hi[i] = base[i] + batt_limit + ev_dis
    return lo, hi

def certify_feeder_constraints(case, margin=1e-8):
    """Safe global screening: returns per-shared-label interval upper bound on
    maximum constraint violation. Only negative *certified* bounds are removed.
    MW and squared-pu rows are compared in their native units.
    """
    if case.grid is None:
        raise ValueError('Need original feeder to certify feasibility')
    g, d = case.grid, case.data
    lo,hi = injection_intervals(d)
    f0,L,drop = _feeder_maps(g,list(map(int,d['agent_bus'])))
    a = L[:,:,None] * lo[None,:,:]
    b = L[:,:,None] * hi[None,:,:]
    flow_min = f0[:,None] + np.minimum(a,b).sum(axis=1)
    flow_max = f0[:,None] + np.maximum(a,b).sum(axis=1)
    vm = -drop @ L
    v0 = float(g['root_voltage'])**2 - drop @ f0
    va = vm[:,:,None]*lo[None,:,:]
    vb = vm[:,:,None]*hi[None,:,:]
    v_min = v0[:,None] + np.minimum(va,vb).sum(axis=1)
    v_max = v0[:,None] + np.maximum(va,vb).sum(axis=1)
    lim = float(d['line_limit_mva'])*math.cos(math.pi/8)
    bound = {}
    for k,e in enumerate(g['branches']):
        bound[f'flow_{e}_upper'] = float(np.max(flow_max[k] - lim))
        bound[f'flow_{e}_lower'] = float(np.max(-lim-flow_min[k]))
    for bus in range(g['bus_count']):
        if bus == g['root']:
            continue
        bound[f'voltage_{bus}_upper'] = float(np.max(v_max[bus]-g['voltage_max'][bus]**2))
        bound[f'voltage_{bus}_lower'] = float(np.max(g['voltage_min'][bus]**2-v_min[bus]))
    records = []
    for label in case.shared_labels:
        if label.startswith('edge_'):
            records.append({'Label':label,'Type':'P2P',
                            'Worst possible violation (bound)':np.nan,
                            'Certified redundant':False})
        else:
            if label not in bound:
                raise KeyError(f'Unknown feeder label: {label}')
            records.append({'Label':label,'Type':'flow' if label.startswith('flow_') else 'voltage',
                            'Worst possible violation (bound)':bound[label],
                            'Certified redundant':bound[label] < -margin})
    return pd.DataFrame(records)

In [8]:
def aggregate_time_grid(data, target_dt=2.0):
    d = copy.deepcopy(data)
    old_dt = float(d['dt'])
    factor = int(round(target_dt/old_dt))
    if factor < 1 or not np.isclose(factor*old_dt,target_dt):
        raise ValueError('New dt must be an integer multiple of original dt')
    oldT = len(d['buy_price'])
    if oldT % factor or int(d['T_eval']) % factor:
        raise ValueError('Horizon and evaluation length must be divisible by aggregation')
    N = len(d['demand']); T=oldT//factor
    def agent_agg(key,op):
        a = np.asarray(d[key],float).reshape(N,T,factor)
        d[key]=a.mean(axis=2) if op=='mean' else a.sum(axis=2)
    for k in ('demand','generation','hp_base_power','hp_q_heat'):
        agent_agg(k,'mean')
    agent_agg('ev_driving_load','sum')   # EV trip consumption is ENERGY per bin
    availability=np.asarray(d['ev_availability'],int).reshape(N,T,factor)
    mixed=((availability.min(axis=2) != availability.max(axis=2)))
    d['ev_availability']=availability.min(axis=2).astype(int)
    for k in ('buy_price','sell_price','p2p_price'):
        d[k]=np.asarray(d[k],float).reshape(T,factor).mean(axis=1)
    d['dt']=float(target_dt);d['resolution_h']=float(target_dt)
    d['T']=T;d['T_eval']=int(data['T_eval'])//factor
    return d, mixed

def solve_compact(case, verbose=False):
    result = solve_vgne(case, verbose=verbose)
    result.pop("mp", None)
    result.pop("avi", None)
    return result


In [5]:
data4,mixed4 = aggregate_time_grid(short_data, target_dt=4.0)
assert len(data4['buy_price']) == 6 and np.isclose(data4['dt'],4.)
assert int(data4['T_eval']) == 6
edges4 = [tuple(sorted(e)) for e in data4['edges']]
print('4h steps:',len(data4['buy_price']))
print('Mixed EV availability bins:', int(np.sum(mixed4)))
if np.any(mixed4): print('WARNING: coarse EV scheduling may change physical outcomes/departure timing.')

4h steps: 6
Mixed EV availability bins: 3


## 2 · Rebuild the certified reduced feeder game and prune local rows

The feeder screening depends on **physical injection bounds**, not on the quadratic ridge. We certify it once. The LP redundancy test then removes only local inequalities proven implied by other local inequalities. We use the resulting `base4` as the common physical game for all ridge levels.

In [10]:
from scenarios import make_scenario_horizon
from p2p.setup import load_ieee33

# Build the original 24-hour scenario if necessary
if "short_data" not in globals():
    short_data = make_scenario_horizon(
        "2016-01-07",
        seed=42,
        resolution_h=0.5,
        lookahead_h=0,
    )

# Convert 48 half-hour steps into 6 four-hour steps
data4, mixed4 = aggregate_time_grid(
    short_data,
    target_dt=4.0,
)

if "grid" not in globals():
    grid = load_ieee33()

print("Time resolution:", data4["dt"], "hours")
print("Time steps:", len(data4["buy_price"]))
print("Prosumers:", len(data4["demand"]))
print("EV availability intervals affected:", int(mixed4.sum()))

Time resolution: 4.0 hours
Time steps: 6
Prosumers: 4
EV availability intervals affected: 3


In [11]:
from p2p.setup import to_static_gnep

full4 = build_game(copy.deepcopy(data4), grid=grid, clearing_epsilon_kw=0.0)
report4 = certify_feeder_constraints(full4)
keep4 = report4.loc[~report4['Certified redundant'], 'Label'].tolist()
reduced4, moved4 = rebuild_constraints(
    full4, keep_labels=keep4, clean_locals=True, localize_single_agent=True
)
print('Rows per step:', len(full4.shared_labels), '→', len(reduced4.shared_labels))
print('Feeder certification:')
display(report4.groupby(['Type','Certified redundant']).size().rename('Rows per step').to_frame())

# Original bilateral-clearing reference, without stronger ridge.
sol_reduced4 = solve_compact(reduced4, verbose=False)
ref_u = np.asarray(sol_reduced4['u'], dtype=float)
ref_cost = np.asarray(evaluated_costs(reduced4, ref_u), dtype=float)
print('Reference day-1 EUR / prosumer:', np.round(ref_cost, 6))
print('Full feeder violation of reference:', model_max_violation(full4, ref_u))

Rows per step: 136 → 8
Feeder certification:


,,Rows per step
Type,Certified redundant,
P2P,False,8
flow,True,64
voltage,True,64


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806500
Academic license 2806500 - for non-commercial use only - registered to ch___@ethz.ch
Reference day-1 EUR / prosumer: [11.050777 54.985863  3.321077 10.859313]
Full feeder violation of reference: 3.5416114485542494e-14


In [12]:
def audit_and_prune_local(case, *, apply=False, tol=1e-7):
    g=case.game
    kw={name:getattr(g,name) for name in GAME_FIELDS}
    newA=[]; newb=[]; audit=[]
    for t in range(case.T):
        a_t=[]; b_t=[]
        for i in range(case.N):
            A=np.asarray(g.C_loc[t][i],float).copy()
            b=np.asarray(g.b_loc[t][i],float).copy()
            initial=len(b)
            keep=list(range(initial))
            failures=0
            for j in range(initial):
                if j not in keep: continue
                others=[k for k in keep if k!=j]
                if not others: continue
                # No variable bounds here; all physical bounds are in A.
                lp=linprog(-A[j],A_ub=A[others],b_ub=b[others],
                           bounds=[(None,None)]*A.shape[1],method='highs')
                if lp.status==0 and (-lp.fun <= b[j]+tol):
                    keep.remove(j)
                elif lp.status not in (0,3):
                    failures+=1  # Infeasible/numerical/other: do not drop
            audit.append(dict(t=t,prosumer=f'P{i+1}',
                  initial=initial,remaining=len(keep),removed=initial-len(keep),
                  LP_unresolved=failures))
            a_t.append(A[keep].copy()); b_t.append(b[keep].copy())
        newA.append(a_t);newb.append(b_t)
    report=pd.DataFrame(audit)
    if not apply: return case,report
    kw.update(C_loc=newA,b_loc=newb)
    return replace(case,game=DynLQGameTV(**kw)),report

start=time.perf_counter()
_,local_report4=audit_and_prune_local(reduced4,apply=False)
print('Local LP audit time:',round(time.perf_counter()-start,2),'s')
display(local_report4.groupby('prosumer')[['initial','remaining','removed','LP_unresolved']].sum())
print('Local rows:',local_report4.initial.sum(),'→',local_report4.remaining.sum())

Local LP audit time: 4.05 s


,initial,remaining,removed,LP_unresolved
prosumer,,,,
P1,192,144,48,0
P2,192,148,44,0
P3,192,132,60,0
P4,192,144,48,0


Local rows: 768 → 568


In [13]:
base4, local_audit = audit_and_prune_local(reduced4, apply=True)
base_ref = solve_compact(base4, verbose=False)
base_u = np.asarray(base_ref['u'], float)
base_cost = np.asarray(evaluated_costs(base4, base_u), float)
print('Local inequalities:', sum(len(b) for bt in reduced4.game.b_loc for b in bt),
      '→', sum(len(b) for bt in base4.game.b_loc for b in bt))
print('Cost Δ from local pruning [EUR]:', np.max(np.abs(base_cost-ref_cost)))
print('Dispatch RMS Δ:', np.sqrt(np.mean((base_u-ref_u)**2)))
print('Violation in full feeder game:', model_max_violation(full4, base_u))
assert base4.T == 6 and base4.N == 4
assert all(s.startswith('edge_') for s in base4.shared_labels)

Local inequalities: 768 → 568
Cost Δ from local pruning [EUR]: 3.300439921360976e-10
Dispatch RMS Δ: 7.457556546190256e-11
Violation in full feeder game: 1.7763568394002505e-15


## 3 · Set up the one-community approximation, preserving all four original trade edges

The original equality pairs are unchanged in `base4`. The grouped model replaces all eight stage-wise bilateral inequalities with **two community-wide inequalities** and adds a penalty for each original bilateral mismatch. This yields `m_sh=12` and `β=36` for six steps. `ρ=100` is the candidate supported by the earlier fidelity table.

In [14]:
from p2p.setup import unpack_controls, evaluated_costs, to_static_gnep
from dynect import DynLQGameTV
from dataclasses import replace
import importlib

edges = [tuple(sorted(map(int, e))) for e in base4.data['edges']]
labels = list(base4.shared_labels)
pos_indices = []
for a,b in edges:
    pos, neg = f'edge_{a}_{b}_pos', f'edge_{a}_{b}_neg'
    assert pos in labels and neg in labels, f'Missing reciprocity pair for edge {a}-{b}'
    ip, im = labels.index(pos), labels.index(neg)
    for t in range(base4.T):
        for i in range(base4.N):
            assert np.allclose(base4.game.C_u[t][i][ip], -base4.game.C_u[t][i][im])
    pos_indices.append(ip)
print('Cyclic edges:', edges)
print('Original β:', (base4.N-1) * base4.T * base4.game.m_u)

Cyclic edges: [(0, 1), (1, 2), (2, 3), (0, 3)]
Original β: 144


In [15]:
GAME_FIELDS = ('A','B','Q','R','P','c','q','r','p','C_x','b_x','C_loc','b_loc','C_u','b_u')

def make_grouped_game(base, edge_groups, *, eps=0.01, rho=0.0):
    """New approximate GNEP: group *signed* reciprocity residuals, with optional bilateral penalty.

    edge_groups: tuple/list of tuples of edge indices, partitioning 0..E-1.
    """
    E = len(edges)
    flat = [k for group in edge_groups for k in group]
    if sorted(flat) != list(range(E)):
        raise ValueError('Groups must partition the original trading edges')
    if eps < 0 or rho < 0:
        raise ValueError('eps and rho must be nonnegative')
    g = base.game
    kw = {name: getattr(g, name) for name in GAME_FIELDS}
    C_new, b_new = [], []
    for t in range(base.T):
        blocks = []
        for i in range(base.N):
            rows = []
            for group in edge_groups:
                c = np.zeros(g.nu[i])
                for edge_idx in group:
                    c += np.asarray(g.C_u[t][i][pos_indices[edge_idx]], float)
                rows.extend((c, -c))
            blocks.append(np.vstack(rows))
        C_new.append(blocks)
        b_new.append(np.full(2 * len(edge_groups), float(eps)))
    kw['C_u'], kw['b_u'] = C_new, b_new

    if rho:
        # R[t][i][j] is d^2 J_i/(du_i du_j), not the entire social-cost Hessian.
        R = [[[np.asarray(g.R[t][i][j], float).copy() for j in range(base.N)]
              for i in range(base.N)] for t in range(base.T)]
        dt = float(base.data['dt'])
        for t in range(base.T):
            for a,b in edges:
                ia, ib = base.controls[a][f'trade_{b}'], base.controls[b][f'trade_{a}']
                # Each endpoint penalizes its bilateral imbalance.
                R[t][a][a][ia,ia] += dt * rho
                R[t][b][b][ib,ib] += dt * rho
                R[t][a][b][ia,ib] += dt * rho
                R[t][b][a][ib,ia] += dt * rho
        kw['R'] = R

    names = [f'group_{k}_{side}' for k in range(len(edge_groups)) for side in ('pos','neg')]
    return replace(base, game=DynLQGameTV(**kw), shared_labels=names,
                   choices={**base.choices, 'grouping': [list(g) for g in edge_groups],
                            'clearing_epsilon_kw': eps, 'mismatch_penalty_rho': rho})

def edge_residuals(case, u):
    """Physical bilateral mismatch, shape (edge, time), kW."""
    controls = unpack_controls(case, u)
    return np.asarray([controls[a][f'trade_{b}'] + controls[b][f'trade_{a}']
                       for a,b in edges], float)

print('Approximation helper ready; no solver has been called.')

Approximation helper ready; no solver has been called.


## 4 · Increase the exact existing quadratic regularizer

The P2P adapter uses `R[t][i][i] += 2 * dt * input_regularization * I`. We add only the **difference** from the baseline coefficient to the existing own-input Hessian blocks. Physical constraints, time aggregation, trading fees, and the `ρ` penalty are unaffected.

For additional confidence, the cell checks the modified Hessian blocks against a **fresh game assembled with `build_game(..., input_regularization=1e-3)`**.

In [16]:
RIDGE_VALUES = (1e-9, 1e-4, 1e-3, 1e-2)
GROUP_RHO = 100.0
GROUP_EPS = 0.01


def set_input_regularization(case, new_reg):
    new_reg = float(new_reg)
    old_reg = float(case.choices.get('input_regularization', 1e-9))
    if new_reg <= 0:
        raise ValueError('Require strictly positive input regularization')
    delta = 2.0 * float(case.data['dt']) * (new_reg - old_reg)
    g = case.game
    kw = {name: getattr(g, name) for name in GAME_FIELDS}
    R = [[[np.asarray(g.R[t][i][j], float).copy() for j in range(case.N)]
          for i in range(case.N)] for t in range(case.T)]
    for t in range(case.T):
        for i in range(case.N):
            R[t][i][i] += delta * np.eye(g.nu[i])
    kw['R'] = R
    return replace(case, game=DynLQGameTV(**kw),
                   choices={**case.choices, 'input_regularization': new_reg})

# Validate that the ridge adjustment matches a fresh adapter assembly.
assembled = build_game(copy.deepcopy(data4), grid=grid,
                       clearing_epsilon_kw=0.0, input_regularization=1e-3)
reweighted = set_input_regularization(base4, 1e-3)
max_R_diff = max(float(np.max(np.abs(
    assembled.game.R[t][i][j] - reweighted.game.R[t][i][j]
))) for t in range(base4.T) for i in range(base4.N) for j in range(base4.N))
print('Objective Hessian vs freshly assembled 1e-3 game, max |Δ|:', f'{max_R_diff:.3e}')
assert max_R_diff < 1e-8, 'Input Hessian manipulation did not match source adapter'

ridge_cases = {}
for reg in RIDGE_VALUES:
    bilateral = set_input_regularization(base4, reg)
    grouped = make_grouped_game(bilateral,
                                (tuple(range(len(edges))),),
                                eps=GROUP_EPS, rho=GROUP_RHO)
    ridge_cases[(reg, 'bilateral')] = bilateral
    ridge_cases[(reg, 'community')] = grouped

print('Built', len(ridge_cases), 'games. No Julia solver has run.')

Objective Hessian vs freshly assembled 1e-3 game, max |Δ|: 0.000e+00
Built 8 games. No Julia solver has run.


## 5 · Conditioning, vGNE accuracy and bilateral clearing residuals

For each ridge coefficient, the notebook solves the bilateral reference and the community approximation at the **same ridge level**. It also compares each to the original (`1e-9`) bilateral model. `evaluated_costs()` reports economic day-1 costs, excluding the numerical ridge and mismatch penalty.

**Crucial:** Cholesky success and eigenvalues diagnose the pseudo-gradient used by DyNECT, but do not establish that region enumeration will be fast. Lack of progress despite positive definiteness points to preprocessing, degeneracy, or other numerical issues.

In [17]:
from scipy.linalg import eigh

summary_rows, ridge_solutions = [], {}

def conditioning(case):
    _, avi, gnep = to_static_gnep(case)
    H = np.asarray(avi.H, float)
    S = H + H.T
    vals = np.linalg.eigvalsh(S)
    try:
        np.linalg.cholesky(S)
        chol = True
    except np.linalg.LinAlgError:
        chol = False
    return dict(n=sum(gnep.n), shared=gnep.m_sh,
                beta=(gnep.N-1)*gnep.m_sh,
                min_eig=float(vals[0]), max_eig=float(vals[-1]),
                cond_est=(float(vals[-1]/vals[0]) if vals[0] > 0 else np.inf),
                cholesky=chol,
                symmetry_error=float(np.max(np.abs(H-H.T))))

for reg in RIDGE_VALUES:
    for kind in ('bilateral', 'community'):
        game = ridge_cases[(reg, kind)]
        tic = time.perf_counter()
        try:
            diag = conditioning(game)
            result = solve_compact(game, verbose=False)
            elapsed = time.perf_counter()-tic
            ridge_solutions[(reg,kind)] = result
            u = np.asarray(result['u'], float)
            c = np.asarray(evaluated_costs(game, u), float)
            mismatch = edge_residuals(game,u)
            reference_same_ridge = (
                u if kind == 'bilateral'
                else np.asarray(ridge_solutions[(reg,'bilateral')]['u'],float)
            )
            ref_same_cost = (
                c if kind == 'bilateral'
                else np.asarray(evaluated_costs(ridge_cases[(reg,'bilateral')], reference_same_ridge),float)
            )
            row = dict(ridge=reg, model=kind, **diag,
                       max_cost_delta_vs_base_EUR=float(np.max(np.abs(c-base_cost))),
                       max_cost_delta_vs_same_ridge_bilateral_EUR=float(np.max(np.abs(c-ref_same_cost))),
                       control_rms_vs_base=float(np.sqrt(np.mean((u-base_u)**2))),
                       control_rms_vs_same_ridge_bilateral=float(np.sqrt(np.mean((u-reference_same_ridge)**2))),
                       max_bilateral_mismatch_kW=float(np.max(np.abs(mismatch))),
                       unmatched_energy_kWh=float(game.data['dt'] * np.sum(np.abs(mismatch))),
                       original_game_max_violation=float(model_max_violation(base4,u)),
                       solve_and_diagnostics_s=elapsed)
            summary_rows.append(row)
            print(f'{kind:10s} ridge={reg:.0e}: β={diag["beta"]}, '
                  f'λmin={diag["min_eig"]:.3e}, '
                  f'Cholesky={diag["cholesky"]}, '
                  f'cost Δ={row["max_cost_delta_vs_base_EUR"]:.5f} EUR, '
                  f'pair Δ={row["max_bilateral_mismatch_kW"]:.6f} kW')
        except Exception as exc:
            summary_rows.append(dict(ridge=reg,model=kind,error=f'{type(exc).__name__}: {exc}'))
            print(f'FAILED {kind} ridge={reg:.0e}: {type(exc).__name__}: {exc}')

ridge_table = pd.DataFrame(summary_rows)
display(ridge_table.round(7))
print('Reminder: the grouped market is a different game even if its vGNE is close.')

bilateral  ridge=1e-09: β=144, λmin=1.600e-08, Cholesky=True, cost Δ=0.00000 EUR, pair Δ=0.000000 kW
community  ridge=1e-09: β=36, λmin=1.600e-08, Cholesky=True, cost Δ=0.00101 EUR, pair Δ=0.000675 kW
bilateral  ridge=1e-04: β=144, λmin=1.600e-03, Cholesky=True, cost Δ=0.10645 EUR, pair Δ=0.000000 kW
community  ridge=1e-04: β=36, λmin=1.600e-03, Cholesky=True, cost Δ=0.10795 EUR, pair Δ=0.000699 kW
bilateral  ridge=1e-03: β=144, λmin=1.600e-02, Cholesky=True, cost Δ=2.28517 EUR, pair Δ=0.000000 kW
community  ridge=1e-03: β=36, λmin=1.600e-02, Cholesky=True, cost Δ=2.28624 EUR, pair Δ=0.000820 kW
bilateral  ridge=1e-02: β=144, λmin=1.600e-01, Cholesky=True, cost Δ=2.44114 EUR, pair Δ=0.000000 kW
community  ridge=1e-02: β=36, λmin=1.600e-01, Cholesky=True, cost Δ=2.44320 EUR, pair Δ=0.001752 kW


,ridge,model,n,shared,beta,min_eig,max_eig,cond_est,cholesky,symmetry_error,max_cost_delta_vs_base_EUR,max_cost_delta_vs_same_ridge_bilateral_EUR,control_rms_vs_base,control_rms_vs_same_ridge_bilateral,max_bilateral_mismatch_kW,unmatched_energy_kWh,original_game_max_violation,solve_and_diagnostics_s
0,0.0000,bilateral,264,48,144,0.0000,0.016001,1.000070e+06,True,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.072189
1,0.0000,community,264,12,36,0.0000,1600.016000,1.000050e+11,True,0.0,0.001011,0.001011,0.000272,0.000272,0.000674,0.060622,0.000674,0.072480
2,0.0001,bilateral,264,48,144,0.0016,0.017601,1.100069e+01,True,0.0,0.106451,0.000000,0.032244,0.000000,0.000000,0.000000,0.000000,0.089765
3,0.0001,community,264,12,36,0.0016,1600.017600,1.000011e+06,True,0.0,0.107948,0.001497,0.032258,0.000348,0.000699,0.061618,0.000699,0.155577
4,0.0010,bilateral,264,48,144,0.0160,0.032001,2.000069e+00,True,0.0,2.285165,0.000000,0.500713,0.000000,0.000000,0.000000,0.000000,0.089254
5,0.0010,community,264,12,36,0.0160,1600.032000,1.000020e+05,True,0.0,2.286238,0.001241,0.500715,0.000330,0.000820,0.067474,0.000820,0.108948
6,0.0100,bilateral,264,48,144,0.1600,0.176001,1.100007e+00,True,0.0,2.441139,0.000000,0.743907,0.000000,0.000000,0.000000,0.000000,0.235665
7,0.0100,community,264,12,36,0.1600,1600.176000,1.000110e+04,True,0.0,2.443199,0.002060,0.743934,0.000551,0.001752,0.120005,0.001752,0.302619


Reminder: the grouped market is a different game even if its vGNE is close.


## 6 · Choose the model / ridge to enumerate

Start with the **community model at `1e-3`**. If it still shows no region progress, change `SELECT_RIDGE` to `1e-2`. You can also choose `SELECT_MODEL = 'bilateral'` for comparison. Do not call Julia if the selected vGNE failed to solve or the curvature check shows a clearly negative eigenvalue.

In [18]:
SELECT_RIDGE = 1e-3         # Change to 1e-2 for the next run
SELECT_MODEL = 'community'  # 'bilateral' or 'community'

key = (float(SELECT_RIDGE), SELECT_MODEL)
assert key in ridge_solutions, f'No vGNE solve available for {key}'
gne_case_small = ridge_cases[key]
matched = ridge_table[(ridge_table['ridge'] == SELECT_RIDGE) &
                      (ridge_table['model'] == SELECT_MODEL)]
assert len(matched) == 1
print('=== SELECTED GAME ===')
print(matched.iloc[0].to_string())
print('Time steps:', gne_case_small.T)
print('Local rows:', sum(len(b) for bt in gne_case_small.game.b_loc for b in bt))
print('β dimension:', (gne_case_small.N-1)*gne_case_small.T*gne_case_small.game.m_u)

=== SELECTED GAME ===
ridge                                                 0.001
model                                             community
n                                                       264
shared                                                   12
beta                                                     36
min_eig                                               0.016
max_eig                                            1600.032
cond_est                                      100002.000003
cholesky                                               True
symmetry_error                                          0.0
max_cost_delta_vs_base_EUR                         2.286238
max_cost_delta_vs_same_ridge_bilateral_EUR         0.001241
control_rms_vs_base                                0.500715
control_rms_vs_same_ridge_bilateral                 0.00033
max_bilateral_mismatch_kW                           0.00082
unmatched_energy_kWh                               0.067474
original_game_max_

## 7 · `GNESetSolver` (unmodified Julia, verbose log, **timeout only**)

Set `RUN_ENUMERATION=True` and execute this cell. It starts a new worker using the **same Python interpreter** as the notebook and explicitly provides both the P2P and DyNECT source paths. It prints newly written log lines every 15 seconds and forces termination after `TIMEOUT_MIN`. It does **not** apply the earlier `get_ignore_masks()` patch, set memory limits, or introduce any other protections.

A region limit controls how many critical regions can be explored, **not how long initialization can take**. If Julia is silent, the notebook still prints a heartbeat; this does not imply that a region has been found.

# Extract AS0 as warm start

In [22]:
import numpy as np
from p2p.setup import to_static_gnep

# Same model and regularization intended for GNE enumeration
case = gne_case_small

# Reuse its computed vGNE
result = ridge_solutions[(float(SELECT_RIDGE), SELECT_MODEL)]
u_star = np.asarray(result["u"], dtype=float).ravel()

mp, avi, gnep = to_static_gnep(case)

A = np.asarray(avi.A, dtype=float)
b = np.asarray(avi.b, dtype=float).ravel()

slack = b - A @ u_star
TOL = 1e-6

active = np.flatnonzero(np.abs(slack) <= TOL)

print("=== vGNE ACTIVE-SET DIAGNOSTIC ===")
print("Decisions:", len(u_star))
print("Total inequalities:", len(b))
print("Active inequalities:", len(active))
print("Active-row rank:", np.linalg.matrix_rank(A[active]))
print("Maximum constraint violation:", max(0.0, -slack.min()))
print("Minimum slack:", slack.min())
print("Shared inequalities:", gnep.m_sh)
print("β dimension:", (gnep.N - 1) * gnep.m_sh)

print("\nActive constraint indices (Python, zero-based):")
print(active.tolist())

print("\nJulia-style indices (one-based):")
print((active + 1).tolist())

=== vGNE ACTIVE-SET DIAGNOSTIC ===
Decisions: 264
Total inequalities: 670
Active inequalities: 316
Active-row rank: 218
Maximum constraint violation: 2.9672152512816297e-09
Minimum slack: -2.9672152512816297e-09
Shared inequalities: 12
β dimension: 36

Active constraint indices (Python, zero-based):
[87, 93, 96, 97, 98, 99, 100, 101, 105, 107, 109, 110, 111, 112, 113, 118, 121, 124, 125, 129, 131, 133, 134, 135, 136, 137, 139, 142, 145, 148, 149, 153, 155, 157, 158, 159, 160, 161, 163, 166, 168, 172, 173, 177, 179, 181, 182, 183, 184, 185, 187, 190, 193, 196, 197, 201, 203, 205, 206, 207, 208, 209, 211, 214, 217, 220, 221, 225, 227, 229, 230, 231, 232, 233, 235, 238, 241, 244, 245, 249, 251, 253, 254, 255, 256, 257, 258, 259, 263, 265, 268, 269, 273, 275, 277, 278, 279, 280, 281, 282, 283, 287, 289, 292, 293, 297, 299, 301, 303, 305, 306, 307, 311, 313, 318, 319, 323, 325, 327, 329, 331, 332, 333, 337, 339, 344, 345, 349, 351, 353, 354, 355, 356, 357, 358, 359, 363, 365, 368, 369, 373,

In [23]:
import inspect
import importlib

gm = importlib.import_module("dynect.solvers.gne")

print("=== GNESetSolver source ===")
print(inspect.getsource(gm.GNESetSolver.__init__))

print("\n=== Julia wrapper ===")
print("Object:", gm._julia.mpsolve)

try:
    print("Signature:", inspect.signature(gm._julia.mpsolve))
except (TypeError, ValueError):
    print("Signature unavailable")

try:
    print(inspect.getsource(gm._julia.mpsolve))
except (TypeError, OSError):
    print("Source unavailable through inspect")

=== GNESetSolver source ===
    def __init__(self, gnep: StaticGNEP, *, beta_lb=None, beta_ub=None, gamma_lb=None, gamma_ub=None,
                 verbose: int = 1, region_limit: int | None = None):
        self.gnep = gnep
        self.mpavi = MpAVI.nabetani(gnep, beta_lb=beta_lb, beta_ub=beta_ub, gamma_lb=gamma_lb, gamma_ub=gamma_ub)
        opts = {"verbose": int(verbose)}
        if region_limit is not None:
            opts["region_limit"] = int(region_limit)
        p = self.mpavi
        raw, status = _julia.mpsolve(p.H, p.F, p.f, p.A, p.B, p.b, p.C, p.d, p.lb, p.ub, opts=opts)
        self.status = _PDAQP_STATUS.get(status, Status.NOT_SOLVED)
        if status != "Solved":
            warnings.warn(f"[GNESetSolver] ParametricDAQP returned status {status}: the GNE set may be incomplete")
        regions = denormalised_regions(raw) if raw is not None else []
        self.gne_set = gne_regions(regions, gnep)


=== Julia wrapper ===
Object: <function mpsolve at 0x15f774ca0>
Signatu

In [19]:
import os, sys, json, time, pickle, signal, subprocess, importlib
from pathlib import Path
from datetime import datetime

RUN_ENUMERATION = True       # Explicitly change to True to launch Julia
BETA = 5.0
REGION_LIMIT = 100
TIMEOUT_MIN = 30
VERBOSE = 2

if not RUN_ENUMERATION:
    print('Enumeration disabled. Inspect the ridge table above; then set RUN_ENUMERATION=True.')
else:
    case = gne_case_small
    nb_expected = (case.N-1)*case.T*case.game.m_u
    p2p_root = Path(importlib.import_module('p2p.setup').__file__).resolve().parents[1]
    dynect_root = Path(importlib.import_module('dynect').__file__).resolve().parents[1]
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(p2p_root),str(dynect_root),env.get('PYTHONPATH','')])
    env['PYTHONUNBUFFERED'] = '1'

    folder = p2p_root/'.gne_experiments'/datetime.now().strftime(
        f'ridge_{SELECT_MODEL}_{SELECT_RIDGE:g}_%Y%m%d_%H%M%S_%f')
    folder.mkdir(parents=True, exist_ok=False)
    input_file, worker_file = folder/'game.pkl', folder/'worker.py'
    log_file, summary_file, out_file = folder/'solver.log', folder/'summary.json', folder/'gne_set.pkl'
    with input_file.open('wb') as f:
        pickle.dump(case,f)
    worker_file.write_text(r'''
import sys,json,time,pickle,traceback
from pathlib import Path
import numpy as np
from p2p.setup import to_static_gnep
from dynect.solvers.gne import GNESetSolver

def log(*xs): print(time.strftime('%H:%M:%S'), *xs, flush=True)
input_file,summary_file,out_file=map(Path,sys.argv[1:4])
beta=float(sys.argv[4]); region_limit=int(sys.argv[5]); verbose=int(sys.argv[6])
try:
    log('Worker Python started; loading game...')
    with input_file.open('rb') as f: case=pickle.load(f)
    tic=time.monotonic()
    log('Condensing to StaticGNEP...')
    _, _, gnep=to_static_gnep(case)
    nb=(gnep.N-1)*gnep.m_sh
    log('Condensed in',round(time.monotonic()-tic,3),'s; decisions=',sum(gnep.n),
        'shared=',gnep.m_sh,'beta=',nb)
    log('ENTER GNESetSolver; region_limit=',region_limit,'beta=±',beta,'verbose=',verbose)
    tic=time.monotonic()
    solver=GNESetSolver(gnep,beta_lb=np.full(nb,-beta),beta_ub=np.full(nb,beta),
                        verbose=verbose,region_limit=region_limit)
    log('GNESetSolver RETURNED after',round(time.monotonic()-tic,2),'s')
    summary=dict(status=str(solver.status),gne_regions=len(solver.gne_set.regions),
                 beta_dimension=nb,region_limit=region_limit)
    summary_file.write_text(json.dumps(summary,indent=2))
    with out_file.open('wb') as f: pickle.dump(solver.gne_set,f)
    log('RESULT',summary)
except BaseException as e:
    log('WORKER ERROR',repr(e));traceback.print_exc()
    summary_file.write_text(json.dumps({'error':repr(e)},indent=2))
    sys.exit(1)
''')

    print(f'RUN: {SELECT_MODEL} / ridge={SELECT_RIDGE:.0e} / β={nb_expected}', flush=True)
    print(f'Julia: limit={REGION_LIMIT}, bounds=±{BETA}, verbose={VERBOSE}, '
          f'timeout={TIMEOUT_MIN} min',flush=True)
    print('Experiment folder:',folder,flush=True)
    started, last_report, last_offset = time.monotonic(), time.monotonic(), 0
    timed_out = False
    with log_file.open('w') as output:
        proc = subprocess.Popen([sys.executable,'-u',str(worker_file),str(input_file),
            str(summary_file),str(out_file),str(BETA),str(REGION_LIMIT),str(VERBOSE)],
            cwd=str(p2p_root),env=env,stdout=output,stderr=subprocess.STDOUT,
            start_new_session=True)
        while proc.poll() is None:
            time.sleep(0.5)
            elapsed=time.monotonic()-started
            if elapsed >= TIMEOUT_MIN*60:
                timed_out = True
                print('TIMEOUT: terminating solver process group',flush=True)
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                break
            if time.monotonic()-last_report >= 15:
                print(f'[{elapsed/60:.2f} min] Worker running...',flush=True)
                with log_file.open(errors='replace') as f:
                    f.seek(last_offset)
                    chunk=f.read()
                    last_offset=f.tell()
                if chunk: print(chunk[-5000:],flush=True)
                else: print('(No new solver messages)',flush=True)
                last_report=time.monotonic()
        proc.wait()
    print('\n=== FINISHED ===')
    print('Exit:',proc.returncode,'| timed out:',timed_out,
          '| elapsed min:',round((time.monotonic()-started)/60,2))
    if summary_file.exists(): print('Summary:',summary_file.read_text())
    else: print('No completed GNE result.')
    if out_file.exists(): print('Saved GNE set:',out_file)
    print('Log tail:\n',log_file.read_text(errors='replace')[-5000:])
    print('Experiment folder:',folder)

RUN: community / ridge=1e-03 / β=36
Julia: limit=100, bounds=±5.0, verbose=2, timeout=30 min
Experiment folder: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/.gne_experiments/ridge_community_0.001_20261125_083234_710811
[0.26 min] Worker running...
08:32:37 Worker Python started; loading game...
08:32:37 Condensing to StaticGNEP...
08:32:37 Condensed in 0.032 s; decisions= 264 shared= 12 beta= 36
08:32:37 ENTER GNESetSolver; region_limit= 100 beta=± 5.0 verbose= 2

[0.51 min] Worker running...
(No new solver messages)
[0.76 min] Worker running...
(No new solver messages)
[1.01 min] Worker running...
(No new solver messages)
[1.27 min] Worker running...
(No new solver messages)
[1.52 min] Worker running...
(No new solver messages)
[1.78 min] Worker running...
(No new solver messages)
[2.03 min] Worker running...
(No new solver messages)
[2.28 min] Worker running...
(No new solver messages)
[2.54 min] Worker running...
(No new solver messages)
[2.79 mi

KeyboardInterrupt: 

## How to interpret the experiment

- If `min eig(H+Hᵀ)` is comfortably positive for `1e-3` / `1e-2` but Julia remains stuck, simply adding curvature is **not** enough; investigate `mpsolve` initialization/preprocessing and the condensed inequality geometry.
- If the community ridge model runs, verify **actual GNE regions** and unilateral best-response gaps before making fairness claims.
- If the larger ridge substantially changes day-1 costs or dispatch, it is a model approximation and should be reported as such.
- The Julia subprocess is subject to a **time limit only**; monitor system memory independently.